In [58]:
import pandas as pd
from joblib import load
import pickle
import polars as pl

from src.config import CONFIG
from src.featuring import nutrisc_algo, encode_category

import gc
gc.collect()

feats = CONFIG['feats_model']
target = CONFIG['target']


path_data = './data/processed/nutrient_cleaned.parquet'
df = pl.read_parquet(path_data)
df = nutrisc_algo(df, 'nutriscore_score')

n = df.shape[0]
print(df.shape)

cols_print = ['nutriscore_score_grade', 'product_name', 'brands_tags']#, 'countries_tags']
X_all = df.select(feats+cols_print)
print(X_all.columns)

# encode cat si besoin
cols = ['tag_0', 'tag2_0']
for col in cols:
    if col in X_all.columns:
        X_all = encode_category(X_all, c=col, new_data=True)
        
# create sample
X_all = X_all.sample(n=2000, seed=21)
# sve to viz
if False:
    X_all.write_parquet("./data/food_toviz.parquet")

print('X_all size ', X_all.shape)
print('Sample size ', df.shape)

(904070, 31)
['fat', 'saturated-fat', 'sugars', 'proteins', 'salt', 'fiber', 'fruits-vegetables-nuts', 'is_fats_nuts_seeds', 'energy-kcal-revise', 'nutriscore_score_grade', 'product_name', 'brands_tags']
X_all size  (2000, 12)
Sample size  (904070, 31)


In [41]:
X_all.select(pl.col('nutriscore_score_grade').value_counts())

nutriscore_score_grade
struct[2]
"{""C"",434}"
"{""D"",530}"
"{""E"",504}"
"{""A"",332}"
"{""B"",200}"


### Carte stats par pays

In [38]:
import pycountry
from babel import Locale
import plotly.express as px

# Dataframe par pays
# res = df['country_0'].value_counts().sort(by='count')

# list_country = res.filter(pl.col('count') > 100)['country_0'].to_list()

# df_country = df.filter(
#         pl.col('country_0').is_in(list_country)
#         )
# df_country['country_0'].value_counts().sort(by='count')

df_country = (
    df.group_by("country_0")
      .agg([
          pl.col("nutriscore_score").mean().alias("nutriscore_moyen"),
          pl.col("nutriscore_score").median().alias("nutriscore_mediane"),
          pl.col("nutriscore_score").count().alias("nb_produits")
          ])
      .sort("nutriscore_moyen")
      )

locale = Locale("fr")
country_to_iso3 = {locale.territories.get(c.alpha_2, c.name): c.alpha_3 for c in pycountry.countries}
df_country = df_country.with_columns(
    pl.col("country_0")
      .map_elements(lambda x: country_to_iso3.get(x, None))
      .alias("country_iso3")
      )

# Save
df_country.write_parquet("./data/nutrisc_country.parquet")

palette_nutriscore = [
    (0.0, "#009E3A"),  # A
    (0.20, "#7ED321"),  # B
    (0.52, "#FFEB3B"),  # C
    (0.84, "#FFA500"),  # D
    (1.0, "#E60000")    # E
]

fig = px.choropleth(
    df_country,
    locations="country_iso3",  # Colonne des pays (ISO alpha-3)
    color="nutriscore_mediane",
    hover_name="country_iso3",  # Nom affiché au survol
    # hover_data=["top_products"],  # Produits représentatifs
    color_continuous_scale=palette_nutriscore,  #px.colors.sequential.YlGnBu,
    title="Qualité nutritionnelle médiane par pays",
)
fig.update_layout(
    geo=dict(
        bgcolor="beige",           # fond derrière la carte
        lakecolor="beige",         # si tu veux aussi les lacs
        showland=True,
        landcolor="beige",         # couleur des terres
        showocean=True,
        oceancolor="lightblue",
        showframe=False,
        projection_type="natural earth"
    ),
    paper_bgcolor="beige",
    margin={"r":0,"t":0,"l":0,"b":0}  # gauche, droite, haut, bas
    )
# fig.update_geos(
#     projection_type="natural earth",
# )

fig.show()

In [39]:
df_country

country_0,nutriscore_moyen,nutriscore_mediane,nb_produits,country_iso3
str,f64,f64,u32,str
"""Monde""",4.371951,2.0,2460,null
"""Europe""",5.275862,4.0,29,null
"""Brésil""",6.900455,3.0,1758,"""BRA"""
"""Venezuela""",6.943396,3.0,53,"""VEN"""
"""Pérou""",7.384615,5.0,104,"""PER"""
…,…,…,…,…
"""Corée du Sud""",13.572519,14.0,131,"""KOR"""
"""Égypte""",14.464789,18.0,71,"""EGY"""
"""Dominique""",15.5,15.5,2,"""DMA"""


### SHAP

In [54]:
import shap
import matplotlib.pyplot as plt

# Charger le pipeline
mod = "nutriments"
pipeline_model = load(f'./model/lgb_pipeline_2025_11_17_{mod}.joblib')
model = pipeline_model.named_steps["lgb"]

# SHAP
X_np = X_all.to_numpy()  # .select(feats)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_np)

# shap.summary_plot(shap_values, X_all)
fig = plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, features=X_np, 
                  feature_names=X_all.columns, show=False)
plt.savefig(f'./output/shap_summary_{mod}.png', dpi=300, bbox_inches='tight')
plt.close()

### Focus un produit spécifique et faire varier la quantité

In [87]:
# Les informations nutritives sont données pour 100g ou 100ml
# et 'fruits-vegetables-nuts' est en % pour 100g

mod = "nutriments"
pipeline_model = load(f'./model/lgb_pipeline_2025_11_17_{mod}.joblib')

# Les aliments D
sc = 'E'
df_aliment = X_all.filter(pl.col("nutriscore_score_grade") == sc)#.to_dicts()[0]
print(df_aliment.shape)

colonne100g = ['fat', 
               'saturated-fat', 
               'sugars', 
               'proteins',
               'salt',
               'fiber',
               'energy-kcal-revise'
               ]


quantites = [20, 40, 60, 80, 100]
# qte_target = 50
# facteur = qte_target/100
df_aliment = df_aliment.with_columns(
    pl.arange(0, df_aliment.height).alias("id_aliment")
)

df_list = []
for qte in quantites:
    facteur = qte / 100
    df_tmp = df_aliment.with_columns(
        [(pl.col(c) * facteur).alias(c) for c in colonne100g] +
        [pl.lit(qte).alias("quantite")] +
        [pl.when(pl.lit(qte) != 100)
            .then(None)
            .otherwise(pl.col("nutriscore_score_grade"))
            .alias("nutriscore_score_grade")]
    )
    df_list.append(df_tmp)

df_variante = pl.concat(df_list, rechunk=True)

df_variante.head(2)

(504, 12)


fat,saturated-fat,sugars,proteins,salt,fiber,fruits-vegetables-nuts,is_fats_nuts_seeds,energy-kcal-revise,nutriscore_score_grade,product_name,brands_tags,id_aliment,quantite
f32,f32,f32,f32,f32,f32,f32,i8,f64,str,str,list[str],i64,i32
7.0,2.8,0.16,2.6,0.4,0.0,0.0,0,74.4,null,"""Leberwurst""","[""xx:meisterklasse""]",0,20
0.6,0.6,4.4,0.2,0.00254,0.2,0.0,0,28.0,null,"""Chocolat Dipped Oranges""","[""torn-ranch""]",1,20


In [88]:
X_aliment = df_variante.select(feats)
print(X_aliment.shape)

# model
pipeline_model = load('./model/lgb_pipeline_2025_11_17_nutriments.joblib')

y_pred = pipeline_model.predict(X_aliment)
df_variante = df_variante.with_columns(pl.Series("nutriscore_score_pred", y_pred))
df_variante = nutrisc_algo(df_variante, 'nutriscore_score_pred')

# save
if False:
    df_variante.write_parquet("./data/aliments_quantity.parquet")

# col_print = feats + cols_print
# row = df.sample(n=1).select(col_print).to_dicts()[0]

(2520, 9)


c:\Users\lilin\AppData\Local\pypoetry\Cache\virtualenvs\foodapp-WvfPqYev-py3.11\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



In [89]:
import random

# ids = df_variante.select("id_aliment").unique().to_series().to_list()
# id_aleatoire = random.choice(ids)

aliment_aleatoire = df_variante.filter(
    (pl.col("id_aliment") == id_aleatoire) 
    & (pl.col("quantite") == 20)
    )

ligne_dict = aliment_aleatoire.to_dicts()[0]

In [90]:
df_produit = pd.DataFrame({
    "Features": list(ligne_dict.keys()),
    "Values": list(ligne_dict.values())
})
df_produit

,Features,Values
0,fat,5.6
1,saturated-fat,2.2
2,sugars,7.0
3,proteins,1.2
4,salt,0.24384
5,fiber,0.98
6,fruits-vegetables-nuts,0.0
7,is_fats_nuts_seeds,0
8,energy-kcal-revise,100.0
9,nutriscore_score_grade,None
